<a href="https://colab.research.google.com/github/tyniquex/big-data-analytics-isda-130/blob/chore%2Fongoing-assignment/classifiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6 · Classifiers

**3 points · due Sunday · ISDA 140, Week 7**

You will train classifiers on your own text, score them on rows they never saw, and put that score beside
the number you wrote in Project 1.

**What you hand in:** this notebook, with the 3 cards answered on **your own** dataset and the checklist at
the bottom ticked. **These 3 cards are the classification half of Project 2**, so
the work carries forward.

## Two halves

The **top half** is a demo on food products, typed in this week's walkthrough video. Read each cell, run it,
read the note under it. The **bottom half** is 3 cards on your own file, with no code given: copy from the
demo, or do it another way.

## The demo data

**USDA FoodData Central, Branded Foods, April 2026**, the same release as Labs 4 and 5. Public domain (US
federal data); no account, no key. The setup cell downloads it for you. For each product it keeps three
things, **the ingredient list, the product name and 8 numbers from the nutrition label**, plus the aisle
the product is filed under (`category`, the 10 most common).


In [ ]:
import sys, pandas as pd, sklearn, duckdb
print(sys.version.split()[0], pd.__version__, sklearn.__version__, duckdb.__version__)

---
## Setup · you do not type this one

Gets the demo data and builds one table, `foods`. **Run it and wait.**

It downloads the release (449 MB), unpacks the 3 files the demo needs, pulls 8 numbers per product out of
the 1,496 MB nutrient file **without loading that file** (Week 6's DuckDB move), and keeps **50,000
products** from the 10 most common aisles. Every product kept has all 8 numbers, an ingredient list and a
name.

In [ ]:
# --- supplied setup. You do not need to read this cell. -------------------------------------------
import os, time, shutil, zipfile, urllib.request
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay, recall_score, precision_score
from sklearn.utils.multiclass import type_of_target

OUT = os.environ.get("ISDA140_OUT", "/content/isda140")    # the same folder Labs 4 and 5 used
os.makedirs(OUT, exist_ok=True)
URL = "https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_branded_food_csv_2026-04-30.zip"
NEED = ["branded_food.csv", "food.csv", "food_nutrient.csv"]
t0 = time.time()
if not all(os.path.exists(f"{OUT}/{n}") for n in NEED):
    ZIP = f"{OUT}/branded.zip"
    if not os.path.exists(ZIP):    # an ordinary User-Agent: some servers refuse urllib's default one
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0 (ISDA 140 course notebook)"})
        with urllib.request.urlopen(req, timeout=900) as r, open(ZIP, "wb") as fh:
            shutil.copyfileobj(r, fh)
    with zipfile.ZipFile(ZIP) as z:
        for info in z.infolist():
            if os.path.basename(info.filename) in NEED:
                with z.open(info) as src, open(f"{OUT}/{os.path.basename(info.filename)}", "wb") as dst:
                    shutil.copyfileobj(src, dst)

# 8 numbers per 100 g, one row per product, straight from the 1,496 MB file
IDS = {1008: "kcal", 1003: "protein_g", 1004: "fat_g", 1005: "carb_g",
       2000: "sugar_g", 1079: "fiber_g", 1093: "sodium_mg", 1253: "cholesterol_mg"}
NUM = list(IDS.values())
pick = ", ".join(f"max(amount) FILTER (WHERE nutrient_id = {k}) AS {v}" for k, v in IDS.items())
nutrients = duckdb.sql(f"""SELECT fdc_id, {pick} FROM '{OUT}/food_nutrient.csv'
                           WHERE nutrient_id IN ({', '.join(map(str, IDS))}) GROUP BY fdc_id""").df()

bf = pd.read_csv(f"{OUT}/branded_food.csv", usecols=["fdc_id", "ingredients", "branded_food_category"])
names = pd.read_csv(f"{OUT}/food.csv", usecols=["fdc_id", "description"])
top10 = bf["branded_food_category"].value_counts().index[:10]
foods = (bf[bf["branded_food_category"].isin(top10)]
         .merge(names, on="fdc_id", how="left")
         .merge(nutrients, on="fdc_id", how="inner")
         .dropna(subset=NUM + ["ingredients", "description"]))
foods = foods[(foods["ingredients"].str.strip() != "") & (foods["description"].str.strip() != "")]
print(f"products with all 8 numbers, ingredients and a name: {len(foods):,}")
foods = (foods.sample(50_000, random_state=140)
              .rename(columns={"branded_food_category": "category", "description": "name"}))
foods["both"] = foods["name"] + " | " + foods["ingredients"]
print(f"foods: {foods.shape[0]:,} rows x {foods.shape[1]} columns   ({time.time() - t0:.0f} s)")

---
# Demo half

**10 steps, typed in the walkthrough video.** Each is a few lines.

| steps | lesson |
|---|---|
| 1–2 | Lesson 1 · what classification is, and a score you can trust |
| 3–6 | Lesson 2 · classifying from numbers |
| 7–10 | Lesson 3 · classifying from text |

### 1 · What kind of answer is it?

`category` is a label: predicting it is **classification**. `kcal` is a number: predicting it would be
**regression**. Look at what scikit-learn thinks `kcal` is.

In [ ]:
print(foods["category"].value_counts(normalize=True).round(3))
print(foods["kcal"].nunique(), type_of_target(foods["kcal"]))

### 2 · Train-test split, and the do-nothing score

Cut the rows in two before looking at any model: 75% to learn from, 25% kept aside. `stratify` keeps each
aisle's share the same on both sides. Then the first score is a model that **does nothing**: it always
answers the biggest aisle.

In [ ]:
tr, te = train_test_split(foods.index, test_size=0.25, random_state=140, stratify=foods["category"])
y_tr, y_te = foods.loc[tr, "category"], foods.loc[te, "category"]
X_tr, X_te = foods.loc[tr, NUM], foods.loc[te, NUM]
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print(len(tr), len(te), round(dummy.score(X_te, y_te), 3))

### 3 · A tree you can read

8 numbers from the label, nothing else. A decision tree asks yes/no questions about them. Keep it 3
questions deep and print it.

In [ ]:
small = DecisionTreeClassifier(max_depth=3, random_state=140).fit(X_tr, y_tr)
print(export_text(small, feature_names=NUM))
print(round(small.score(X_te, y_te), 3))

### 4 · Let it grow, then let many trees vote

No depth limit this time, then a random forest (many trees, each a little different, voting). Score each
twice: on the rows it learned from, and on the rows it never saw.

In [ ]:
big = DecisionTreeClassifier(random_state=140).fit(X_tr, y_tr)
forest = RandomForestClassifier(random_state=140, n_jobs=-1).fit(X_tr, y_tr)
for name, m in [("tree", big), ("forest", forest)]:
    print(name, round(m.score(X_tr, y_tr), 3), round(m.score(X_te, y_te), 3))
print(big.get_depth(), big.get_n_leaves())

### 5 · The confusion matrix

Rows are the true aisle, columns are what the forest said. The diagonal is right; everything else is a
mistake. Then read the loudest mistake as real rows.

In [ ]:
pred = forest.predict(X_te)
ConfusionMatrixDisplay.from_predictions(y_te, pred, xticks_rotation=90)
test = foods.loc[te].assign(pred=pred)
test.query("category == 'Candy' and pred == 'Chocolate'")[["name", "sugar_g", "fat_g"]].head(3)

### 6 · Is it Chocolate?

Ask one question only. A model that always answers "no" is right on every non-chocolate row. How many real
chocolates does each model find?

In [ ]:
print("always 'no' :", round((y_te != "Chocolate").mean(), 3))
print("found       :", ((y_te == "Chocolate") & (pred == "Chocolate")).sum(), "of", (y_te == "Chocolate").sum())
print("recall      :", round(recall_score(y_te, pred, labels=["Chocolate"], average=None)[0], 3))
print("precision   :", round(precision_score(y_te, pred, labels=["Chocolate"], average=None)[0], 3))

### 7 · Words as features

Same products, same split, but now the model reads the ingredient list. Week 4's TF-IDF turns each list into
a row of numbers, **fitted on the training rows only**, then applied to the test rows.

In [ ]:
vec = TfidfVectorizer()
W_tr = vec.fit_transform(foods.loc[tr, "ingredients"])
W_te = vec.transform(foods.loc[te, "ingredients"])
print(W_tr.shape, W_te.shape)

### 8 · Four classifiers, the same rows, one table

The two tree models plus Naive Bayes and logistic regression, on the same split. The table has accuracy,
recall for the smallest aisle, and how long each took.

In [ ]:
smallest = y_tr.value_counts().idxmin()
models = {"naive bayes": MultinomialNB(), "logistic regression": LogisticRegression(max_iter=1000),
          "decision tree": DecisionTreeClassifier(random_state=140),
          "random forest": RandomForestClassifier(random_state=140, n_jobs=-1)}
rows = []
for name, m in models.items():
    t = time.time(); m.fit(W_tr, y_tr); secs = time.time() - t
    p = m.predict(W_te)
    rows.append([name, m.score(W_te, y_te), recall_score(y_te, p, labels=[smallest], average=None)[0], secs])
print("smallest aisle:", smallest)
pd.DataFrame(rows, columns=["model", "accuracy", "smallest recall", "seconds"]).round(3)

### 9 · One change at a time: what the model reads

Keep logistic regression, keep the split, change only the text: the ingredients, the product name, or both.

In [ ]:
for col in ["ingredients", "name", "both"]:
    v = TfidfVectorizer()
    A, B = v.fit_transform(foods.loc[tr, col]), v.transform(foods.loc[te, col])
    print(col, round(LogisticRegression(max_iter=1000).fit(A, y_tr).score(B, y_te), 3))

### 10 · The same text on both sides of the split

The same ingredient list can belong to several products. If one copy is in the training rows and another is
in the test rows, the model is graded on something it has already seen.

In [ ]:
print(round(foods.loc[te, "ingredients"].isin(foods.loc[tr, "ingredients"]).mean(), 3))
uniq = foods.drop_duplicates(subset="ingredients")
tr2, te2 = train_test_split(uniq.index, test_size=0.25, random_state=140, stratify=uniq["category"])
v = TfidfVectorizer()
A, B = v.fit_transform(uniq.loc[tr2, "ingredients"]), v.transform(uniq.loc[te2, "ingredients"])
print(len(uniq), round(LogisticRegression(max_iter=1000).fit(A, uniq.loc[tr2, "category"]).score(B, uniq.loc[te2, "category"]), 3))

---
# Your turn · 3 cards on your own file

**3 cards, 1 point each.** No code is given. Copy from the demo or do it another way. Add as many cells as
you want under each card.

You use **your own** dataset: the file from Project 0, **the text column you named in Lab 1**, and **the
target you locked in Project 1**.

## TODO 1 · your file, your text column, your target

Copy `DATASET_CHOICE` and `TEXT_COLUMN` from your Lab 3 notebook, and `TARGET_COLUMN` from your Project 1.

In [ ]:
# TODO 1: fill in all five, between the quotes.
DATASET_CHOICE = {
    "name":    "REPLACE ME",     # a short name you will recognise later
    "source":  "REPLACE ME",     # a URL, a Drive link, the name of a file you uploaded, or hf:owner/name
    "doc_url": "REPLACE ME",     # the page that says how the data was collected
}
TEXT_COLUMN   = "REPLACE ME"     # the free-text column (exact spelling)
TARGET_COLUMN = "REPLACE ME"     # the column you predict, as locked in Project 1

assert "REPLACE ME" not in (*DATASET_CHOICE.values(), TEXT_COLUMN, TARGET_COLUMN), (
    "Fill in TODO 1 before running the cards.")

In [ ]:
# --- supplied loader (same load_any as Labs 1 to 3). You do not need to read this cell. -------------
assert DATASET_CHOICE["source"] != "REPLACE ME", "Fill in TODO 1 above and run it first."
import io, re

_DRIVE_RE = re.compile(r"drive\.google\.com/(?:file/d/|open\?id=|uc\?(?:[^&]*&)*id=)([\w-]+)")

def _read_drive(file_id):
    with urllib.request.urlopen(f"https://drive.google.com/uc?export=download&id={file_id}", timeout=180) as r:
        blob = r.read()
    if blob[:64].lstrip().lower().startswith((b"<!doctype", b"<html")):
        raise RuntimeError("Drive sent a web page, not the file. Share it as 'Anyone with the link'.")
    if blob[:2] == b"\x1f\x8b": return pd.read_csv(io.BytesIO(blob), compression="gzip", low_memory=False)
    if blob[:4] == b"PAR1":     return pd.read_parquet(io.BytesIO(blob))
    return pd.read_csv(io.BytesIO(blob), low_memory=False)

def load_any(src):
    if src.startswith("hf:"):
        from datasets import load_dataset
        return load_dataset(src[3:], split="train").to_pandas()
    m = _DRIVE_RE.search(src)
    if m: return _read_drive(m.group(1))
    low = src.lower().split("?")[0]
    if low.endswith(".parquet"):          return pd.read_parquet(src)
    if low.endswith((".json", ".jsonl")): return pd.read_json(src, lines=low.endswith(".jsonl"))
    return pd.read_csv(src, low_memory=False)

df = load_any(DATASET_CHOICE["source"])
for col in (TEXT_COLUMN, TARGET_COLUMN):
    assert col in df.columns, f"{col!r} is not a column in this file. Its columns: {list(df.columns)[:15]}"
print(f"{DATASET_CHOICE['name']}: {len(df):,} rows")
print(df[TARGET_COLUMN].value_counts(dropna=False).head(15))

## Card 1 · Train-test split and the do-nothing score

1. **In one sentence: is your target a classification or a regression question, and why?** (Demo step 1:
   a column of whole numbers can be read as labels without any warning.)
2. Drop rows with no text or no label, and apply the decision you made in Project 1 about small classes.
   **If more than 20,000 rows are left, take one random sample of 20,000 and use it for all 3 cards:**
   `df = df.sample(20_000, random_state=140)`.
3. Cut the rows 75/25 (stratified, a fixed `random_state`). Compute the **do-nothing score** and put it beside
   the number you wrote in Project 1. **If they differ, say why.**
4. Train logistic regression on TF-IDF of your text (fitted on the training rows only). Report its score on
   the **training** rows and on the **test** rows.
5. Count how many of your test texts also appear in your training rows. Then drop repeated texts
   (`drop_duplicates` on your text column) **before** splitting, score again, and report both scores.

**Hand in:** the sentence from step 1, the numbers from steps 3 to 5, and 2 to 3 sentences on what they say.

> A file balanced by construction gives a do-nothing score of about 1 divided by the number of classes
> (a random sample is never perfectly even). That is a correct result.

In [ ]:
# Card 1 · your code (add cells as you like)

**Your sentences:**

## Card 2 · Model evaluation

Draw the **confusion matrix** for your logistic regression from card 1. Name the pair of labels it confuses
most, and copy **2 of those rows** (their text and both labels). Report **precision and recall for your
smallest class**. Then say **which number you would report** for your file, and why.

**Hand in:** the matrix and 3 to 4 sentences.

> If your smallest class is never predicted, scikit-learn prints `UndefinedMetricWarning` and its precision
> shows as 0. That is not an error; it is your finding: that class's recall is 0.

In [ ]:
# Card 2 · your code

**Your sentences:**

## Card 3 · Classifier comparison

Train **Naive Bayes, logistic regression, a decision tree and a random forest** on the same rows and the
same split. Make a table: **accuracy, recall for your smallest class, fit seconds** (demo step 8). **Print the
top two levels of your tree** and say what it asks first (demo step 3 shows `export_text`; for words the
names come from your vectorizer's `get_feature_names_out()`, and `max_depth=2` keeps it short; the lines that
say `truncated branch of depth ...` are the parts it hides, and the depth tells you how big your full tree grew). Then **choose
one model for Project 2** and say why, with a reason besides its number being highest.

**Hand in:** the table, what your tree asks first, and 2 to 3 sentences on your choice.

> **On long review text the trees often lose badly and logistic regression wins. That is a correct result**,
> not a failed lab. And on a file of a few thousand rows, a gap of a few points between two models can come
> from the split alone.

In [ ]:
# Card 3 · your code

**Your sentences:**

---
## This week's readings

1. **Inria scikit-learn MOOC, "Classification"**, from the top to the end of "The issue of class imbalance".
   Read it; do not paste its code into Colab (its data path only works on its own site).
   <https://inria.github.io/scikit-learn-mooc/python_scripts/metrics_classification.html>
2. **Google Machine Learning Crash Course, "Dividing the original dataset"**, the section on test rows that
   repeat training rows. <https://developers.google.com/machine-learning/crash-course/overfitting/dividing-datasets#additional_problems_with_test_sets>

Reference, if you need a function's details (the **1.6** pages, which match Colab):
[classification metrics](https://scikit-learn.org/1.6/modules/model_evaluation.html#classification-metrics) ·
[data leakage](https://scikit-learn.org/1.6/common_pitfalls.html#data-leakage)

---
## If you see this error

**`ValueError: np.nan is an invalid document, expected byte or unicode string.`**
Some rows have no text. Drop them before splitting (card 1, step 2), and say how many.

**`ValueError: The least populated class in y has only 1 member, which is too few. ...`**
`stratify` needs at least 2 rows of every class. Apply your Project 1 decision about small classes first:
merge them or drop them.

**No error, but a classifier with dozens of classes you did not expect**
Your target is a column of whole numbers (a score, a rating, a year), and scikit-learn read every value as its
own class. Card 1's first sentence is where you decide whether that is what you want. Only numbers with
decimals (like 87.5) stop with `Unknown label type: continuous`.

**`ValueError: Input y contains NaN.`**
Some rows have no label. Drop them before splitting.

**`ConvergenceWarning: lbfgs failed to converge ... Increase the number of iterations (max_iter) or scale the data`**
Use `LogisticRegression(max_iter=1000)`. Ignore the part about scaling: it is the wrong fix for TF-IDF.

**`UndefinedMetricWarning: Precision is ill-defined ...`**
Not an error. One of your classes was never predicted, so its precision has nothing to divide by. That is a
finding for card 2.

**A tree model runs for minutes**
A decision tree on long texts is slow. The 20,000-row sample in card 1 keeps it to seconds.

**`KeyError`, or `... is not a column in this file`**
Check the spelling in TODO 1 against the column list the loader printed.

**Fewer rows than your dataset's page says**
For `hf:` sources the loader reads only the `train` part. That is fine for this lab; say which rows your score
is on.

---
## Before you submit

- [ ] The setup cell ran and printed the number of products
- [ ] Demo steps 1–10 all ran
- [ ] TODO 1 is filled in with my own file, text column and target
- [ ] Card 1: my sentence on classification vs regression; the do-nothing score beside my Project 1 number
      (and why they differ, if they do); training and test scores; the repeated-text count and the re-score
- [ ] Card 2: the confusion matrix, the pair and its 2 rows, precision and recall for my smallest class, and
      the number I would report
- [ ] Card 3: all 4 models on the same rows and split, the table, what my tree asks first, and my choice with
      a reason
- [ ] Runtime → Restart and run all, with no errors above TODO 1